## Experiment Parser

this takes the experiment json and pulls all needed information to retrun python varibales
also handles backend validation for selecting a backend 



In [1]:
#imports 
import json 
from qiskit_aer import AerSimulator

#imports for backend validation 
import os
from dotenv import load_dotenv
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService


## backend validation 

In [2]:

import os
from dotenv import load_dotenv
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService


def get_backend(backend_name):
    """
    @brief Validates and retrieves a quantum computing backend.

    @param backend_name Name of the requested backend.
    @return Dictionary containing backend object, type, and name.
    """

    # Validate backend name
    if not isinstance(backend_name, str):
        raise TypeError("Backend name must be a string.")

    backend_name = backend_name.strip()

    if not backend_name:
        raise ValueError("Backend name cannot be empty.")

    #defualt simulator
    if backend_name == "aer_simulator":
        return {
            "backend": AerSimulator(),
            "type": "simulator",
            "name": backend_name
        }

    #confirm quantum computer selection
    #potentially move to testing_framework() to allow parser to work without input hold
    confirmation = input(
        f"You selected to use the quantum computer '{backend_name}'. "
        "Are you sure you want to continue? (Y/N): "
    ).strip().upper()

    if confirmation != "Y":
        raise PermissionError(
            "Quantum computer selection cancelled."
        )

    # Load IBM Quantum credentials
    load_dotenv()

    token = os.getenv("IBM_QUANTUM_API_KEY")
    instance = os.getenv("IBM_QUANTUM_INSTANCE")

    if not token or not instance:
        raise ValueError(
            "IBM Quantum credentials are missing."
        )

    # Connect to IBM Quantum
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=token,
            instance=instance
        )

        #Retrieve accessible real quantum backends
        available_backends = service.backends(simulator=False)

        # Create dictionary of backend names and objects
        backend_list = {
            backend.name: backend
            for backend in available_backends
        }

    except Exception as error:
        raise RuntimeError(
            f"Unable to retrieve IBM Quantum backends: {error}"
        ) from error

    # Validate backend against available backends
    if backend_name not in backend_list:
        raise ValueError(
            f"Invalid or inaccessible backend '{backend_name}'. "
            f"Available backends: {list(backend_list.keys())}"
        )

    # Retrieve validated backend
    backend = backend_list[backend_name]

    # Check backend operational status
    if not backend.status().operational:
        raise RuntimeError(
            f"Backend '{backend_name}' is not operational."
        )

    return {
        "backend": backend,
        "type": "hardware",
        "name": backend_name
    }


In [15]:

import json


def ExperimentParser(json_path):
    """
    @brief Parses and validates experiment configuration.

    @param json_path JSON string or dictionary containing experiment settings.

    @return Dictionary containing validated backend, shots,
            accuracy, and test configuration.
    """
    try:
        with open(json_path, "r") as file:
            experiment = json.load(file)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Experiment JSON file not found: {json_path}"
        )
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON format in: {json_path}"
        ) from error


    # Validate JSON structure
    if not isinstance(experiment, dict):
        raise TypeError("Experiment JSON must be a dictionary.")

    # Extract parameters
    backend_name = experiment.get("backend", "aer_simulator")
    shots = experiment.get("shots", 1000)
    accuracy = experiment.get("accuracy", 0.90)

    # Validate shots
    if type(shots) is not int:
        raise TypeError("Shots must be an integer.")

    if shots <= 0:
        raise ValueError("Shots must be greater than zero.")

    # Validate accuracy
    if type(accuracy) not in (int, float):
        raise TypeError("Accuracy must be numeric.")

    if not 0 <= accuracy <= 1:
        raise ValueError("Accuracy must be between 0 and 1.")

    accuracy = float(accuracy)
    shots = int(shots)

    # Validate and retrieve backend
    backend_info = get_backend(backend_name)
    backend = backend_info["backend"]

    return backend, shots, accuracy
